Критерии информативности в решающих деревьях измеряют степень неоднородности выборки.

Например, критерий Джини показывает, насколько высок шанс ошибиться, если случайным образом выбрать элемент из узла и так же случайно присвоить ему метку класса на основе распределения в этом же узле. Энтропия Шеннона оценивает этот же хаос через меру неопределенности системы.

Дерево решений использует эти критерии, чтобы находить наилучшие разделения данных, стремясь свести неоднородность в финальных узлах к нулю.


### Индекс Джини

Эта формула измеряет уровень «хаоса» классов в текущем узле

$$G(y) = 1 - \sum_{i=1}^{K} p_i^2$$


$K$ — количество уникальных классов в узле.

$p_{i}$ — доля (вероятность) объектов $i$-го класса в этом узле. Она рассчитывается как:

$$p_i = \frac{n_i}{N}$$

где:
* $n_{i}$ — количество объектов класса $i$
* $N$ — общее количество объектов в узле


### Энтропия Шеннона (Shannon Entropy)

Это альтернативная мера хаоса, которую мы добавили для гибкости. Она сильнее штрафует модель за неопределенность за счет использования логарифма.

$$H(y) = - \sum_{i=1}^{K} p_i \log_2(p_i)$$

Где:

* $p_i$ — точно такая же доля объектов $i$-го класса в узле.
* $\log_2$ — логарифм по основанию 2 (если все объекты делятся ровно 50/50, энтропия будет равна ровно 1).
* Знак **минус** в начале обязателен, так как логарифм от дробной вероятности всегда отрицательный, а мера хаоса должна быть положительной.


### Формула прироста информации (Information Gain)

Это главная целевая функция вашего алгоритма. Модель перебирает все признаки и пороги, чтобы **максимизировать** эту величину при каждом разбиении.

$$Gain = I(y_{parent}) - \left( \frac{N_{left}}{N_{parent}} \cdot I(y_{left}) + \frac{N_{right}}{N_{parent}} \cdot I(y_{right}) \right)$$

Где:

* $I(y)$ — это выбранная метрика хаоса (`Gini` или `Entropy`) для конкретного подмножества данных.
* $y_{parent}$ — данные в исходном узле (до разделения).
* $y_{left}$ и $y_{right}$ — данные, ушедшие в левую и правую ветку после применения маски `X[col] <= mask`.
* $N_{parent}$, $N_{left}$, $N_{right}$ — количество объектов в родительском, левом и правом узлах соответственно.
* Выражение в скобках — это **взвешенный хаос потомков** (пропорционально их размеру).


In [89]:
from collections import Counter
import pandas as pd
import numpy as np

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature        # Назв признака
        self.threshold = threshold    # Порог сплита (mask)
        self.left = left              # Ссылка на левый узел Node
        self.right = right            # Ссылка на правый узел Node
        self.value = value            # Финальный ответ (только если это лист!)
        
    def is_leaf(self):
        return self.value is not None

      
class Decision_tree:
  """
    Реализует алгоритм дерева решений (Decision Tree) для задач классификации.
    
    Поддерживает работу с категориальными и числовыми признаками, критерии 
    информативности Gini и Entropy, а также механизм логирования процесса обучения.
    
    Аргументы инициализации :
    ----------------------------------
    max_depth : int, default=100
        Максимальная глубина дерева. Ограничивает рост дерева для предотвращения 
        переобучения.
    min_samples_split : int, default=2
        Минимальное количество объектов в узле, необходимое для выполнения сплита.
        Задается строго как именованный аргумент.
    criterion : str, default='gini'
        Критерий информативности для оценки качества разбиения.
        Доступные варианты: 'gini' (индекс Джини), 'entropy' (энтропия Шеннона).
    log : bool, default=False
        Флаг включения текстового логирования рекурсивного процесса построения дерева.
    
    Основные методы:
    ----------------
    fit(X, y)
        Запускает рекурсивное построение дерева решений на обучающей выборке.
    predict(X)
        Предсказывает метки классов для объектов из переданного датафрейма.
    print_tree()
        Визуализирует структуру обученного дерева в виде текстового графа в консоли.
    
    Доступные критерии информативности (из self.criterion):
    ------------------------------------------------------
    - 'gini'    : Индекс Джини. Измеряет степень неопределенности выборки (критерий по умолчанию).
    - 'entropy' : Энтропия Шеннона. Оценивает хаотичность распределения классов с логарифмом по основанию 2.
    """

  def __init__(self, max_depth = 100, *, min_samples_split=2, criterion = 'gini', log = False):
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.criterion = criterion
    self.root = None
    self.log = log

  def get_gini(self, series: pd.Series):
    """ Расчет джини"""
    len_series = len(series)
    if len_series == 0:
          return 0
    return 1 - sum([(val/len_series)**2 for val in Counter(series).values()])

  def get_entropy(self, series: pd.Series):
        """ Расчет энтропии Шеннона """
        len_series = len(series)
        if len_series == 0:
            return 0
        entropy = 0
        for val in Counter(series).values():
            p = val / len_series 
            entropy -= p * np.log2(p)  # Формула Шеннона с логарифмом по основанию 2
        return entropy

  def _get_metog(self, series: pd.Series):
        """ Вспомогательный метод для выбора нужного критерия """
        if self.criterion == 'entropy':
            return self.get_entropy(series)
        return self.get_gini(series)

  def fit(self,X,y):
      self.root = self._build_tree(X, y, depth = 0)
      return self

  def _build_tree(self,X, y, depth):
    num_samples = len(X)

    log_indent = "   " * depth 
    if self.log:
      print(f"{log_indent}📥 [Вход в рекурсию] Глубина: {depth}, Объектов в узле: {num_samples}")

    if len(Counter(y)) == 1:
      val = y.iloc[0]
      if self.log:  
        print(f"{log_indent}   🍃 СТОП: Все объекты класса {val}. Лист создан")
      return Node(value=val)  # Оставшийся класс

    if num_samples < self.min_samples_split:
        most_value_class = Counter(y).most_common(1)[0][0]
        if self.log:
          print(f"{log_indent}   🍃 СТОП: Мало объектов ({num_samples} < {self.min_samples_split}). Голосование большинства -> Класс {most_value_class}")
        return Node(value=most_value_class)

    if depth >= self.max_depth:
      most_value_class = Counter(y).most_common(1)[0][0]
      if self.log:
        print(f"{log_indent}   🍃 СТОП: Достигли max_depth. Голосование большинства -> Класс {most_value_class}")
      return Node(value=most_value_class)

    tot_size = len(X)
    base_gini = self._get_metog(y)

    best_gain = 0.0
    best_split = None
    best_feature = None

    for col in X.columns:
      unique_col = X[col].unique()

      for mask in unique_col:
        left_mask = X[col] <= mask
        right_mask = X[col] > mask

        y_left = y[left_mask]
        y_right = y[right_mask]

        if len(y_left) == 0 or len(y_right) == 0:
            continue

        left_gini = self._get_metog(y_left)
        rigth_gini = self._get_metog(y_right)

        weighted_gini = (len(y_left) / tot_size) * left_gini + (len(y_right) / tot_size) * rigth_gini

        gini_gain = base_gini - weighted_gini

        if gini_gain > best_gain:
                  best_gain = gini_gain
                  best_feature = col
                  best_split = mask

    if best_gain <= 0 or best_feature is None:  # Если нет прироста Джини или лучшего признака
        most_value_class = Counter(y).most_common(1)[0][0]
        if self.log:
          print(f"{log_indent}   🍃 СТОП: Сплит не уменьшает Джини. Голосование большинства -> Класс {most_value_class}")
        return Node(value=most_value_class)

    if self.log:
      print(f"{log_indent}   ✂️ Разделяем по признаку [{best_feature} <= {best_split}]")
    left_split = X[best_feature] <= best_split
    right_split = X[best_feature] > best_split

    X_left, y_left, X_right, y_right = X[left_split], y[left_split], X[right_split], y[right_split]

    if self.log:
      print(f"{log_indent}   👉 Спуск в ЛЕВУЮ ветку...")
    left_node = self._build_tree(X_left, y_left, depth + 1)
    if self.log:
      print(f"{log_indent}   👈 Спуск в ПРАВУЮ ветку...")
    right_node = self._build_tree(X_right, y_right, depth + 1)

    if self.log:
      print(f"{log_indent}📤 [Выход из рекурсии] Узел на глубине {depth} полностью собран.")
    #self.root = Node(feature=best_feature, threshold=best_split, left=left_node, right=right_node)
    return Node(feature=best_feature, threshold=best_split, left=left_node, right=right_node)

  def predict(self,X: pd.DataFrame):
      #return np.array([self._traverse_tree(row, self.root) for _, row in X.iterrows()])
      rows_as_dicts = X.to_dict(orient='records')
      return np.array([self._traverse_tree(row, self.root) for row in rows_as_dicts])
        
  def _traverse_tree(self, x, node: Node):
      # Если пришли в лист — возвращаем значение
      if node.is_leaf():
          return node.value

        # Иначе идем влево или вправо в зависимости от порога
      if x[node.feature] <= node.threshold:      
          return self._traverse_tree(x, node.left)  
      return self._traverse_tree(x, node.right)

  @property
  def print_tree(self):
      """ Для печати дерева с корня """
      if self.root is None:
          print("Дерево еще не обучено!")
      else:
          self._print_node(self.root)

  def _print_node(self, node: Node, depth=0):
      """ Рекурсивный метод обхода для визуализации вывода """
      indent = "    " * depth  # Отступы зависят от текущей глубины узла

      # Если узел является листом
      if node.is_leaf():
          print(f"{indent}🍃 Класс: {node.value}")
          return

      # Печатаем текущую развилку
      print(f"{indent}❓ {node.feature} <= {node.threshold}")

      # Сначала рекурсивно идем в левую ветку (True)
      print(f"{indent}├── Да:")
      self._print_node(node.left, depth + 1)

      # Затем в правую ветку (False)
      print(f"{indent}└── Нет:")
      self._print_node(node.right, depth + 1)

  


In [90]:
print(Decision_tree().__doc__)


Реализует алгоритм дерева решений (Decision Tree) для задач классификации.

Поддерживает работу с категориальными и числовыми признаками, критерии 
информативности Gini и Entropy, а также механизм логирования процесса обучения.

Аргументы инициализации :
----------------------------------
max_depth : int, default=100
    Максимальная глубина дерева. Ограничивает рост дерева для предотвращения 
    переобучения.
min_samples_split : int, default=2
    Минимальное количество объектов в узле, необходимое для выполнения сплита.
    Задается строго как именованный аргумент.
criterion : str, default='gini'
    Критерий информативности для оценки качества разбиения.
    Доступные варианты: 'gini' (индекс Джини), 'entropy' (энтропия Шеннона).
log : bool, default=False
    Флаг включения текстового логирования рекурсивного процесса построения дерева.

Основные методы:
----------------
fit(X, y)
    Запускает рекурсивное построение дерева решений на обучающей выборке.
predict(X)
    Предсказывае

In [91]:
data = {
    'Возраст': [56, 46, 32, 25, 38, 56, 36, 40, 28, 28],
    'Баланс':  [132000, 146000, 64000, 80000, 36000, 61000, 87000, 84000, 20000, 79000],
    'Купил':   [1, 1, 0, 0, 0, 1, 1, 1, 0, 0]
}

df = pd.DataFrame(data)

X_train = df[['Возраст', 'Баланс']]
y_train = df['Купил']

tree = Decision_tree(max_depth=3, min_samples_split=2, criterion='gini', log=True)
tree.fit(X_train, y_train)

📥 [Вход в рекурсию] Глубина: 0, Объектов в узле: 10
   ✂️ Разделяем по признаку [Возраст <= 32]
   👉 Спуск в ЛЕВУЮ ветку...
   📥 [Вход в рекурсию] Глубина: 1, Объектов в узле: 4
      🍃 СТОП: Все объекты класса 0. Лист создан
   👈 Спуск в ПРАВУЮ ветку...
   📥 [Вход в рекурсию] Глубина: 1, Объектов в узле: 6
      ✂️ Разделяем по признаку [Баланс <= 36000]
      👉 Спуск в ЛЕВУЮ ветку...
      📥 [Вход в рекурсию] Глубина: 2, Объектов в узле: 1
         🍃 СТОП: Все объекты класса 0. Лист создан
      👈 Спуск в ПРАВУЮ ветку...
      📥 [Вход в рекурсию] Глубина: 2, Объектов в узле: 5
         🍃 СТОП: Все объекты класса 1. Лист создан
   📤 [Выход из рекурсии] Узел на глубине 1 полностью собран.
📤 [Выход из рекурсии] Узел на глубине 0 полностью собран.


In [92]:
tree.print_tree

❓ Возраст <= 32
├── Да:
    🍃 Класс: 0
└── Нет:
    ❓ Баланс <= 36000
    ├── Да:
        🍃 Класс: 0
    └── Нет:
        🍃 Класс: 1


In [93]:
X_test = pd.DataFrame({
    'Возраст': [50, 20, 35],
    'Баланс':  [100000, 30000, 50000]})
my_tree = tree.predict(X_test)

In [94]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=3, min_samples_split=2, criterion='gini')
model.fit(X_train, y_train)
sklearn_tree = model.predict(X_test)

print(f"""Предсказания Decision_tree {my_tree}
Предсказания DecisionTree из sklearn {sklearn_tree}""")

Предсказания Decision_tree [1 0 1]
Предсказания DecisionTree из sklearn [1 0 1]


### Сравним результат работы алгоритма для задачи мультиклассовой классификации на примере встроенного набора данных Ирисы Фишера (Iris Dataset)

In [95]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import pandas as pd

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target)

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state=42)

param = {'max_depth':3, 'min_samples_split':2, 'criterion':'gini'}

My_model_tree = Decision_tree(**param).fit(X_train, y_train)
My_predict = My_model_tree.predict(X_test)

SK_model_tree = DecisionTreeClassifier(**param).fit(X_train, y_train)
SK_predict = SK_model_tree.predict(X_test)

In [96]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f"""=== СРАВНЕНИЕ ACCURACY ===
Decision_tree :  {accuracy_score(y_test, My_predict):.4f}
Sklearn дерево : {accuracy_score(y_test, SK_predict):.4f}

=== ПОДРОБНЫЙ ОТЧЕТ Decision_tree ===
{classification_report(y_test, My_predict, target_names=iris.target_names)}

=== ПОДРОБНЫЙ ОТЧЕТ Sklearn дерево ===
{classification_report(y_test, SK_predict, target_names=iris.target_names)}

=== МАТРИЦА ОШИБОК Decision_tree ===
{confusion_matrix(y_test, My_predict)}

=== МАТРИЦА ОШИБОК Sklearn дерево ===
{confusion_matrix(y_test, SK_predict)}""")

=== СРАВНЕНИЕ ACCURACY ===
Decision_tree :  0.9333
Sklearn дерево : 1.0000

=== ПОДРОБНЫЙ ОТЧЕТ Decision_tree ===
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        19
  versicolor       1.00      0.77      0.87        13
   virginica       0.81      1.00      0.90        13

    accuracy                           0.93        45
   macro avg       0.94      0.92      0.92        45
weighted avg       0.95      0.93      0.93        45


=== ПОДРОБНЫЙ ОТЧЕТ Sklearn дерево ===
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        19
  versicolor       1.00      1.00      1.00        13
   virginica       1.00      1.00      1.00        13

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45


=== МАТРИЦА ОШИБОК Decision_tree ===
[[19  0  0]
 [ 0 10  3]
 [ 0  0 13]]

=== МАТР

In [97]:
My_model_tree.print_tree

❓ petal length (cm) <= 1.9
├── Да:
    🍃 Класс: 0
└── Нет:
    ❓ petal length (cm) <= 4.7
    ├── Да:
        ❓ petal width (cm) <= 1.5
        ├── Да:
            🍃 Класс: 1
        └── Нет:
            🍃 Класс: 2
    └── Нет:
        ❓ petal width (cm) <= 1.7
        ├── Да:
            🍃 Класс: 2
        └── Нет:
            🍃 Класс: 2


###  Результаты сравнительного анализа моделей

#### 1. Метрики эффективности модели
* **Общая точность (Accuracy):**
  > Разработанное дерево решений продемонстрировало высокую обобщающую способность, достигнув метрики **Accuracy = 0.93** (42 верно предсказанных объекта из 45) на тестовой выборке.

* **Качество по классам:**
  * Алгоритм абсолютно безошибочно выделил класс `setosa` (метрики $Precision$, $Recall$ и $F_1\text{-}score$ равны **1.00**), что подтверждает корректность базовой логики ветвления.
  * Различие в метриках возникло на границе классов `versicolor` и `virginica`. Согласно матрице ошибок, разработанный алгоритм допустил 3 ошибки, отнеся объекты класса `versicolor` к классу `virginica` ($Recall_{versicolor} = 0.77$, $Precision_{virginica} = 0.81$).

---

#### 2. Причины расхождения результатов
* **Поиск порогов:** 
  В разработанном классе пороги сплита выбираются строго из уникальных значений признаков обучающей выборки (`X[col].unique()`). Алгоритм `scikit-learn` использует более хитрую стратегию (вычисляет средние точки между отсортированными значениями).
  
* **Разрешение коллизий:** 
  При совпадении максимального прироста информации ($Gini\ Gain$) для разных признаков кастомная модель выбирает первый попавшийся в цикле признак, в то время как `scikit-learn` применяет дополнительные критерии балансировки весов.
